In [ ]:
import pandas as pd
df = pd.read_csv('clean_data.csv')
df.head(5)

In [ ]:
import pandas as pd

target_col = 'Time_taken(min)'

# 1. Target
y = df[target_col]

# 2. Features (Drop target AND leakage columns)
leakage_cols = [target_col, 'order_making(min)', 'delivery_time(min)']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])

# 3. Extract time features
X['order_hour'] = pd.to_datetime(X['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.hour
X['order_minute'] = pd.to_datetime(X['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.minute
X['picked_hour'] = pd.to_datetime(X['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.hour
X['picked_minute'] = pd.to_datetime(X['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.minute

X = X.drop(columns=['Time_Orderd', 'Time_Order_picked'])

# 4. Extract date features
X['Order_Date'] = pd.to_datetime(X['Order_Date'], dayfirst=True, errors='coerce')
X['order_day'] = X['Order_Date'].dt.day
X['order_month'] = X['Order_Date'].dt.month
X['order_year'] = X['Order_Date'].dt.year

X = X.drop(columns=['Order_Date'])

# 5. One-hot encode categorical columns
X_encoded = pd.get_dummies(
    X,
    columns=[
        'Weatherconditions',
        'Road_traffic_density',
        'Type_of_order',
        'Type_of_vehicle',
        'City',
        'time_category'
    ],
    drop_first=True,
    dtype=int
)

X_encoded.head()

In [ ]:
# @title Default title text
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Verify the shapes
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Define the hyperparameters to test
param_grid = {
    'n_estimators': [100, 200],
    'max_features': [0.7, 1.0],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_depth': [10, 20]
}

# Initialize Random Forest
model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# Train and search
grid_search.fit(X_train, y_train)

# Best model
model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# TRAIN
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# TEST
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("TRAINING SET")
print(f"MAE:  {train_mae:.2f} minutes")
print(f"MSE:  {train_mse:.2f}")
print(f"RMSE: {train_rmse:.2f} minutes")
print(f"R²:   {train_r2:.4f}")

print("\nTEST SET")
print(f"MAE:  {test_mae:.2f} minutes")
print(f"MSE:  {test_mse:.2f}")
print(f"RMSE: {test_rmse:.2f} minutes")
print(f"R²:   {test_r2:.4f}")

# # 1. Predict on the test set
# y_pred = model.predict(X_test)

# # 2. Calculate evaluation metrics
# mae = mean_absolute_error(y_test, y_pred)
# mse = mean_squared_error(y_test, y_pred)
# rmse = np.sqrt(mse)
# r2 = r2_score(y_test, y_pred)

# # 3. Display the results
# print(f"MAE  (Mean Absolute Error):      {mae:.2f} minutes")
# print(f"MSE  (Mean Squared Error):       {mse:.2f}")
# print(f"RMSE (Root Mean Squared Error):  {rmse:.2f} minutes")
# print(f"R² Score:                        {r2:.4f}")